In [28]:
import re
import warnings
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from tqdm import tqdm


class SanitizeLogits(LogitsProcessor):
    """Replace NaN/Inf logits so multinomial sampling never crashes."""
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        return torch.nan_to_num(scores, nan=0.0, posinf=1e4, neginf=-1e4)


In [29]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
G = 4                 # completions per prompt
MAX_NEW = 1024        # max new tokens (more headroom for math reasoning)
LR = 1e-3
EPOCHS = 1
CLIP_EPS = 0.2        # PPO clip epsilon
KL_COEF = 0.01        # KL penalty coefficient
TEMPERATURE = 0.9
TRAIN_SIZE = 50      # GSM8K training subset
EVAL_SIZE = 10        # GSM8K eval subset

print(f"Using device: {DEVICE}")

Using device: mps


In [30]:
# ── Dataset ───────────────────────────────────────────────────────────────────
def extract_answer(text: str) -> str | None:
    """Extract the final numeric answer from a GSM8K solution string.
    GSM8K solutions end with '#### <number>'."""
    match = re.search(r"####\s*([\d,\-]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()
    return None

In [31]:
SYSTEM_PROMPT = (
    "You are a helpful math tutor. Solve problems step by step. "
    "At the end, write your final answer after '####'."
)

def build_messages(question: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

def apply_chat(messages: list[dict]) -> tuple[torch.Tensor, torch.Tensor]:
    """Tokenize a chat conversation and return (input_ids, attention_mask) tensors on DEVICE."""
    template = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
    )
    input_ids = template['input_ids']
    mask = template['attention_mask']

    return torch.tensor([input_ids], dtype=torch.long, device=DEVICE), torch.tensor([mask], dtype=torch.long, device=DEVICE)

In [32]:
print("Loading GSM8K dataset...")
raw = load_dataset("openai/gsm8k", "main")

train_items = [
    {"messages": build_messages(ex["question"]), "answer": extract_answer(ex["answer"])}
    for ex in raw["train"].select(range(TRAIN_SIZE))
    if extract_answer(ex["answer"]) is not None
]
eval_items = [
    {"messages": build_messages(ex["question"]), "answer": extract_answer(ex["answer"])}
    for ex in raw["test"].select(range(EVAL_SIZE))
    if extract_answer(ex["answer"]) is not None
]

print(f"Train samples: {len(train_items)} | Eval samples: {len(eval_items)}")

Loading GSM8K dataset...
Train samples: 50 | Eval samples: 10


In [34]:
# LoRA config
LORA_R = 4
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# ── Model ─────────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Policy model: base weights frozen, only LoRA adapters are trainable
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model = model.to(DEVICE)
model.train()
model.print_trainable_parameters()

# Frozen reference model for KL penalty
ref_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)
ref_model = ref_model.to(DEVICE)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

trainable params: 230,400 || all params: 134,745,408 || trainable%: 0.1710


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [35]:
optimizer = AdamW(model.parameters(), lr=LR)

# ── Reward function ────────────────────────────────────────────────────────────
def reward_fn(completion: str, gold_answer: str) -> float:
    """Multi-signal reward for GSM8K:
    - 1.0  exact match on the '#### <n>' final answer
    - 0.5  gold number appears anywhere in the completion
    - 0.1  completion contains any digit (partial credit)
    - 0.0  otherwise
    """
    pred = extract_answer(completion)
    if pred is not None and pred == gold_answer:
        return 1.0
    if re.search(rf"\b{re.escape(gold_answer)}\b", completion):
        return 0.5
    if re.search(r"\d", completion):
        return 0.1
    return 0.0


In [36]:

_sanitize = SanitizeLogits()

# ── Eval ──────────────────────────────────────────────────────────────────────
@torch.inference_mode()
def evaluate(items: list[dict], tag: str = "eval") -> float:
    """Greedy decode one completion per item, report mean reward."""
    model.eval()
    total_reward = 0.0
    total = len(items)

    print(f"\n{'─'*60}")
    print(f"[{tag}] Evaluating on {total} items...")

    for i, item in tqdm(enumerate(items), total=total, desc=f"[{tag}] Evaluating"):
        input_ids, mask = apply_chat(item["messages"])
        prompt_len = input_ids.shape[1]

        out = model.generate(
            input_ids=input_ids,
            attention_mask=mask,
            max_new_tokens=MAX_NEW,
            do_sample=False,                    # greedy for eval
            pad_token_id=tokenizer.eos_token_id,
            logits_processor=[_sanitize],
        )
        completion = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True)
        gold = item["answer"]
        r = reward_fn(completion, gold)
        total_reward += r

        print(f"  [r={r:.1f}] gold={gold!r} | {completion[:80]!r}")

    mean_reward = total_reward / total
    print(f"[{tag}] Mean reward: {mean_reward:.4f}")
    print(f"{'─'*60}\n")
    model.train()
    return mean_reward

# ── Helpers ───────────────────────────────────────────────────────────────────
@torch.no_grad()
def sample_completions(messages: list[dict], n: int):
    """Sample n completions for a chat prompt in a single batched generate call."""
    model.eval()  # disable dropout for stable probabilities
    try:
        input_ids, mask = apply_chat(messages)
        prompt_len = input_ids.shape[1]

        # Generate all n completions in one batched forward pass
        out = model.generate(
            input_ids=input_ids.repeat(n, 1),
            attention_mask=mask.repeat(n, 1),
            max_new_tokens=MAX_NEW,
            do_sample=True,
            temperature=TEMPERATURE,
            pad_token_id=tokenizer.eos_token_id,
            logits_processor=[_sanitize],
        )  # (n, prompt_len + max_completion_len)

        all_ids = [out[i:i+1] for i in range(n)]
        all_texts = []
        for i in range(n):
            text = tokenizer.decode(out[i, prompt_len:], skip_special_tokens=True)
            all_texts.append(text)
            print(f"  sample {i + 1}/{n}: {text[:120]!r}")

        return all_ids, all_texts, prompt_len
    finally:
        model.train()  # always restore training mode

# ── GRPO step ─────────────────────────────────────────────────────────────────
def grpo_step(messages: list[dict], gold_answer: str):
    # 1. Sample G completions (all in one batched generate)
    all_ids, completions, prompt_len = sample_completions(messages, G)

    # 2. Score with reward function
    rewards = torch.tensor(
        [reward_fn(c, gold_answer) for c in completions],
        dtype=torch.bfloat16, device=DEVICE
    )

    # 3. Normalize rewards within group (GRPO advantage)
    if rewards.std() > 1e-8:
        advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
    else:
        advantages = rewards - rewards.mean()

    total_loss = torch.tensor(0.0, dtype=torch.bfloat16, device=DEVICE, requires_grad=True)
    eos_id = tokenizer.eos_token_id

    for i, (seq_ids, advantage) in enumerate(zip(all_ids, advantages)):
        completion_ids = seq_ids[0, prompt_len:].clone()

        # Strip trailing padding: keep up to and including first EOS
        eos_positions = (completion_ids == eos_id).nonzero(as_tuple=True)[0]
        if len(eos_positions) > 0:
            completion_ids = completion_ids[:eos_positions[0] + 1]

        if len(completion_ids) == 0:
            continue

        # Trim seq_ids to actual length (no padding)
        seq_ids = seq_ids[:, :prompt_len + len(completion_ids)].clone()
        seq_mask = torch.ones_like(seq_ids)

        # 4. Current policy log probs over completion tokens
        logits = model(seq_ids, attention_mask=seq_mask).logits  # (1, seq, vocab)
        completion_logits = logits[0, prompt_len - 1:-1]         # align to completion tokens
        log_probs = F.log_softmax(completion_logits, dim=-1)
        token_log_probs = log_probs.gather(1, completion_ids.unsqueeze(-1)).squeeze(-1)

        # 5. Reference policy log probs (for KL)
        with torch.no_grad():
            ref_logits = ref_model(seq_ids, attention_mask=seq_mask).logits
            ref_completion_logits = ref_logits[0, prompt_len - 1:-1]
            ref_log_probs = F.log_softmax(ref_completion_logits, dim=-1)
            ref_token_log_probs = ref_log_probs.gather(1, completion_ids.unsqueeze(-1)).squeeze(-1)

        # 6. Ratio for PPO-style clipping (sequence-level log prob ratio)
        old_log_prob = token_log_probs.detach().sum()
        new_log_prob = token_log_probs.sum()
        ratio = torch.exp(new_log_prob - old_log_prob)

        # 7. Clipped surrogate objective
        clipped_ratio = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS)
        policy_loss = -torch.min(ratio * advantage, clipped_ratio * advantage)

        # 8. KL penalty: KL(policy || ref) ≈ mean(log_prob - ref_log_prob) per token
        kl = (token_log_probs - ref_token_log_probs).mean()

        loss = policy_loss + KL_COEF * kl
        total_loss = total_loss + loss / G

    return total_loss, rewards, completions


In [42]:
# Sanity check: verify the model responds as a chat model
test_messages = [
    {"role": "user", "content": eval_items[0]["messages"][1]["content"]},
]
input_ids, mask = apply_chat(test_messages)
prompt_len = input_ids.shape[1]
gold = eval_items[0]["answer"]

with torch.inference_mode():
    out = model.generate(
        input_ids=input_ids.repeat(4, 1),
        attention_mask=mask.repeat(4, 1),   # must match repeated input_ids batch dim
        max_new_tokens=2048,
        do_sample=True,
        temperature=TEMPERATURE,
        pad_token_id=tokenizer.eos_token_id,
    )
    responses = [tokenizer.decode(out[i, prompt_len:], skip_special_tokens=True) for i in range(out.shape[0])]
    rewards = [reward_fn(r, gold) for r in responses]


In [ ]:
# ── Reward function ────────────────────────────────────────────────────────────
def reward_fn(completion: str, gold_answer: str) -> float:
    """Rubric based assessment for GSM8K.

    Points are summed and capped at 1.0, then the length penalty is subtracted.

    Correctness:
      +1.00  exact match on the '#### <n>' final answer
      +0.50  gold number appears as a whole word anywhere in the completion

    Format / structure:
      +0.20  uses '#### <answer>' format (even if the answer is wrong)
      +0.15  shows arithmetic work: expression like '3 * 4' or '12 / 3'
      +0.15  shows equation results: writes '= <number>' at least once
      +0.10  answer is within 2× of the gold value (correct order of magnitude)
      +0.05  has 3+ distinct numeric values (multi-step reasoning indicator)
      +0.05  contains any digit (minimum partial credit)

    Penalty:
      -0.20 max  length penalty: scales linearly with word count up to MAX_NEW
    """
    points = 0.0
    pred = extract_answer(completion)

    # ── Correctness ──────────────────────────────────────────────────────────
    if pred is not None and pred == gold_answer:
        points += 1.0
    if re.search(rf"\b{re.escape(gold_answer)}\b", completion):
        points += 0.5

    # ── Format / structure ───────────────────────────────────────────────────
    if pred is not None:
        points += 0.2                               # used #### format

    # Arithmetic expressions: digits on both sides of an operator
    if re.search(r"\d+\s*[+\-*/×÷]\s*\d+", completion):
        points += 0.15

    # Equation results: '= <number>' (model is writing out intermediate steps)
    if re.search(r"=\s*\$?\s*\d+", completion):
        points += 0.15

    # Order-of-magnitude credit: predicted number within 2× of gold
    try:
        gold_val = float(gold_answer)
        pred_val = float(pred) if pred is not None else None
        if pred_val is not None and gold_val != 0 and 0.5 <= pred_val / gold_val <= 2.0:
            points += 0.10
    except (ValueError, ZeroDivisionError):
        pass

    # Multi-step indicator: 3 or more distinct numeric values in the completion
    distinct_nums = set(re.findall(r"\b\d+(?:\.\d+)?\b", completion))
    if len(distinct_nums) >= 3:
        points += 0.05

    if re.search(r"\d", completion):
        points += 0.05                              # minimum partial credit

    # ── Length penalty ───────────────────────────────────────────────────────
    # Word count as a cheap proxy for token count; max penalty 0.2 at MAX_NEW.
    num_words = len(completion.split())
    length_penalty = 0.2 * (num_words / MAX_NEW)

    return max(0.0, min(points, 1.0) - length_penalty)

print("\nSanity Check Completions")
print("=" * 70)
for i, (resp, r) in enumerate(zip(responses, rewards)):
    reward_custom = reward_fn(resp, gold)
    correct = "✓" if reward_custom == 1.0 else "✗"
    print(f"\n[Sample {i + 1}]  Reward: {r:.1f}  Gold: {gold!r}  Custom Reward: {reward_custom:.2f}  {correct}")
    print("─" * 70)
    print(resp.strip())
    print("─" * 70)



Sanity Check Completions

[Sample 1]  Reward: 0.1  Gold: '18'  Custom Reward: 0.1  ✗
──────────────────────────────────────────────────────────────────────
Janet lays 16 eggs per day, which means she spends 16 x 4 = 64 eggs per day.
She needs to sell 1472 eggs per day to cover the cost of buying and baking the muffins and selling the eggs.
To make one fresh duck egg, she spends 3 for breakfast, 2 for a muffin, and 4 for the eggs.
Janet makes a total of 64 x 2 = 132 eggs per day, so she earns $132 / 64 = $2.10 per egg.
So, in dollars, Janet makes $2.10 every day at the farmers' market.
──────────────────────────────────────────────────────────────────────

[Sample 2]  Reward: 0.1  Gold: '18'  Custom Reward: 0.3  ✗
──────────────────────────────────────────────────────────────────────
Janet's ducks lay 16 eggs per day.
She eats three for breakfast every morning and bakes muffins for her friends every day with four.
Janet makes 16 x 3 = 48 eggs per day.
She sells the remainder at the far

In [46]:
pre_acc = evaluate(eval_items, tag="pre-train eval")
print(f"Starting GRPO training... (pre-train eval acc: {pre_acc:.2%})")


────────────────────────────────────────────────────────────
[pre-train eval] Evaluating on 10 items...


[pre-train eval] Evaluating:   0%|          | 0/10 [00:05<?, ?it/s]


KeyboardInterrupt: 

In [26]:
for epoch in range(EPOCHS):
    print(f"\n{'='*60}\nEpoch {epoch + 1}/{EPOCHS}\n{'='*60}")
    epoch_loss = 0.0

    for step, item in tqdm(enumerate(train_items), desc="Training"):
        optimizer.zero_grad()

        print(f"\nStep {step + 1}/{len(train_items)} | gold={item['answer']!r}")
        loss, rewards, completions = grpo_step(item["messages"], item["answer"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        print(f"  rewards: {[round(float(r), 2) for r in rewards.tolist()]}")
        print(f"  loss:    {loss.item():.4f}")

    print(f"\nEpoch avg loss: {epoch_loss / len(train_items):.4f}")


Epoch 1/1


Training: 0it [00:00, ?it/s]


Step 1/50 | gold='72'
  sample 1/4: 'Half of the number of clips sold in April = 48/2 = 24 clips sold\nIn May she sold 24/2 = 12 clips\nTherefore, she sold 12 '
  sample 2/4: 'In April, Natalia sold 48 clips, and in May, she sold 1/4 of the total number of clips she sold in April, which is 48 / '
  sample 3/4: 'In April, Natalia sold 48/2 = 24 clips.\nIn May, Natalia sold the remaining clips together: 48 (from April) + 24 (from Ma'
  sample 4/4: 'In April, Natalia sold 125 clips to 35 friends, so she sold 125/35 = 3 clips.\nIn May, Natalia sold 125/3 = 4 clips.\nIn t'


Training: 1it [00:04,  4.62s/it]

  rewards: [0.1, 0.1, 0.1, 0.1]
  loss:    -0.0000

Step 2/50 | gold='10'
  sample 1/4: "To solve this problem, let's break down the information given:\n\nNumber of hours babysitting she did: 50\nAmount of money "
  sample 2/4: 'Yesterday, Yesterday did 50 minutes of babysitting, so she earned $50 for that one hour.'
  sample 3/4: 'To solve this problem, we need to calculate the total number of hours she worked and then divide it by the total number '
  sample 4/4: 'To find out how much Yesterday, she earned, we first need to calculate how much she babysat for that length of time.\n\nSh'


Training: 2it [00:10,  5.08s/it]

  rewards: [0.1, 0.1, 0.1, 0.1]
  loss:    -0.0000

Step 3/50 | gold='5'


Training: 2it [00:37, 18.98s/it]


RuntimeError: probability tensor contains either `inf`, `nan` or element < 0

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────

# Pre-training eval
pre_acc = evaluate(eval_items, tag="pre-train eval")



# Post-training eval
post_acc = evaluate(eval_items, tag="post-train eval")

print(f"\n{'='*60}")
print(f"Accuracy: {pre_acc:.2%} → {post_acc:.2%} ({post_acc - pre_acc:+.2%})")
print(f"{'='*60}")

print("\nDone! Saving LoRA adapter...")
model.save_pretrained("./grpo-lora")
tokenizer.save_pretrained("./grpo-lora")